<a href="https://colab.research.google.com/github/gabriela0402/pathfolio-AI/blob/main/notebooks/00_environment_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pathfolio — Verificação e ingestão inicial

Este notebook será usado para testar a leitura de textos digitados,
arquivos PDF e arquivos DOCX.


## 1. Preparação do ambiente

Nesta etapa, o Pathfolio receberá informações escritas manualmente e as transformará em uma estrutura padronizada.


In [2]:
from datetime import datetime, timezone

print("Ambiente do Pathfolio funcionando!")


Ambiente do Pathfolio funcionando!


In [3]:
def create_text_source(
    content: str,
    source_name: str,
    category: str = "manual_text"
) -> dict:
    """
    Converte um texto digitado pelo usuário em uma fonte padronizada.
    """

    cleaned_content = content.strip()

    if not cleaned_content:
        raise ValueError("O conteúdo não pode estar vazio.")

    return {
        "source_type": "manual_text",
        "source_name": source_name,
        "category": category,
        "content": cleaned_content,
        "metadata": {
            "created_at": datetime.now(timezone.utc).isoformat(),
            "page": None,
            "file_name": None
        }
    }


In [4]:
manual_source = create_text_source(
    content="""
    Participei de uma iniciação científica envolvendo a comparação de modelos
    de inteligência artificial aplicados à correção de redações. Trabalhei com
    uma base de dados extensa, análise de resultados e comparação de métricas.
    """,
    source_name="Experiência acadêmica",
    category="academic_experience"
)

manual_source


{'source_type': 'manual_text',
 'source_name': 'Experiência acadêmica',
 'category': 'academic_experience',
 'content': 'Participei de uma iniciação científica envolvendo a comparação de modelos\n    de inteligência artificial aplicados à correção de redações. Trabalhei com\n    uma base de dados extensa, análise de resultados e comparação de métricas.',
 'metadata': {'created_at': '2026-11-06T22:12:16.206045+00:00',
  'page': None,
  'file_name': None}}

In [5]:
assert manual_source["source_type"] == "manual_text"
assert manual_source["source_name"] == "Experiência acadêmica"
assert manual_source["category"] == "academic_experience"
assert len(manual_source["content"]) > 0
assert manual_source["metadata"]["page"] is None

print("Teste de texto manual aprovado.")


Teste de texto manual aprovado.


In [6]:
try:
    create_text_source(
        content="   ",
        source_name="Texto vazio"
    )
except ValueError as error:
    print(f"Validação funcionando: {error}")


Validação funcionando: O conteúdo não pode estar vazio.


## 2. Leitura de documentos

Nesta etapa, o Pathfolio será capaz de ler arquivos PDF e DOCX.

Cada documento será convertido para uma ou mais fontes padronizadas,
preservando o nome do arquivo e, quando possível, a página de origem.


In [7]:
!pip -q install pypdf python-docx


In [8]:
from pathlib import Path

from pypdf import PdfReader
from docx import Document
from google.colab import files


In [9]:
def read_pdf(file_path: str) -> list[dict]:
    """
    Extrai o texto de um PDF página por página.
    """

    reader = PdfReader(file_path)
    sources = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        text = text.strip()

        if not text:
            continue

        sources.append({
            "source_type": "pdf",
            "source_name": Path(file_path).name,
            "category": "uploaded_document",
            "content": text,
            "metadata": {
                "created_at": datetime.now(timezone.utc).isoformat(),
                "page": page_number,
                "file_name": Path(file_path).name
            }
        })

    return sources


In [10]:
def read_docx(file_path: str) -> list[dict]:
    """
    Extrai o texto dos parágrafos de um arquivo DOCX.
    """

    document = Document(file_path)
    paragraphs = []

    for paragraph in document.paragraphs:
        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    content = "\n".join(paragraphs).strip()

    if not content:
        return []

    return [{
        "source_type": "docx",
        "source_name": Path(file_path).name,
        "category": "uploaded_document",
        "content": content,
        "metadata": {
            "created_at": datetime.now(timezone.utc).isoformat(),
            "page": None,
            "file_name": Path(file_path).name
        }
    }]


In [11]:
def read_document(file_path: str) -> list[dict]:
    """
    Lê um PDF ou DOCX usando o leitor correspondente.
    """

    extension = Path(file_path).suffix.lower()

    if extension == ".pdf":
        return read_pdf(file_path)

    if extension == ".docx":
        return read_docx(file_path)

    raise ValueError(
        f"Formato não suportado: {extension}. "
        "Envie um arquivo PDF ou DOCX."
    )


In [12]:
uploaded_files = files.upload()


Saving 03_descricao_projeto_ficticio.docx to 03_descricao_projeto_ficticio.docx
Saving 02_relatorio_projeto_ficticio.pdf to 02_relatorio_projeto_ficticio.pdf
Saving 01_curriculo_candidato_ficticio.pdf to 01_curriculo_candidato_ficticio.pdf


In [13]:
uploaded_sources = []

for file_name in uploaded_files.keys():
    document_sources = read_document(file_name)
    uploaded_sources.extend(document_sources)

print(f"Fontes extraídas: {len(uploaded_sources)}")

for source in uploaded_sources:
    print(f"Arquivo: {source['source_name']}")
    print(f"Tipo: {source['source_type']}")
    print(f"Página: {source['metadata']['page']}")
    print(f"Caracteres: {len(source['content'])}")
    print("-" * 50)


Fontes extraídas: 4
Arquivo: 03_descricao_projeto_ficticio.docx
Tipo: docx
Página: None
Caracteres: 807
--------------------------------------------------
Arquivo: 02_relatorio_projeto_ficticio.pdf
Tipo: pdf
Página: 1
Caracteres: 870
--------------------------------------------------
Arquivo: 01_curriculo_candidato_ficticio.pdf
Tipo: pdf
Página: 1
Caracteres: 907
--------------------------------------------------
Arquivo: 01_curriculo_candidato_ficticio.pdf
Tipo: pdf
Página: 2
Caracteres: 522
--------------------------------------------------


In [14]:
all_sources = [manual_source] + uploaded_sources

print(f"Total de fontes: {len(all_sources)}")

for index, source in enumerate(all_sources, start=1):
    print(
        f"{index}. "
        f"{source['source_name']} | "
        f"{source['source_type']} | "
        f"{len(source['content'])} caracteres"
    )


Total de fontes: 5
1. Experiência acadêmica | manual_text | 231 caracteres
2. 03_descricao_projeto_ficticio.docx | docx | 807 caracteres
3. 02_relatorio_projeto_ficticio.pdf | pdf | 870 caracteres
4. 01_curriculo_candidato_ficticio.pdf | pdf | 907 caracteres
5. 01_curriculo_candidato_ficticio.pdf | pdf | 522 caracteres


In [15]:
for source in all_sources:
    preview = source["content"][:300].replace("\n", " ")

    print(f"Fonte: {source['source_name']}")
    print(f"Tipo: {source['source_type']}")
    print(f"Prévia: {preview}")
    print("=" * 80)


Fonte: Experiência acadêmica
Tipo: manual_text
Prévia: Participei de uma iniciação científica envolvendo a comparação de modelos     de inteligência artificial aplicados à correção de redações. Trabalhei com     uma base de dados extensa, análise de resultados e comparação de métricas.
Fonte: 03_descricao_projeto_ficticio.docx
Tipo: docx
Prévia: Descrição de Projeto — Documento de Teste Projeto: Painel de acompanhamento de estudos Contexto Aplicação web fictícia criada para organizar metas de estudo, registrar sessões e sugerir revisões com base no histórico do usuário. Contribuições realizadas Levantamento dos requisitos principais da apli
Fonte: 02_relatorio_projeto_ficticio.pdf
Tipo: pdf
Prévia: Relatório de Projeto — Documento de Teste Projeto: Classificação de feedbacks acadêmicos Responsável: Marina Oliveira Resumo O projeto investigou técnicas de processamento de linguagem natural para organizar feedbacks acadêmicos por tema e prioridade, facilitando a análise por professores. A

In [16]:
assert len(all_sources) >= 1

for source in all_sources:
    assert source["content"].strip()
    assert source["source_name"]
    assert source["source_type"] in {"manual_text", "pdf", "docx"}
    assert "metadata" in source
    assert "file_name" in source["metadata"]

print("Testes de ingestão de documentos aprovados.")


Testes de ingestão de documentos aprovados.


In [17]:
try:
    read_document("arquivo.txt")
except ValueError as error:
    print(f"Validação funcionando: {error}")


Validação funcionando: Formato não suportado: .txt. Envie um arquivo PDF ou DOCX.


## 3. Limpeza e divisão dos conteúdos

Nesta etapa, os conteúdos extraídos de textos manuais, PDFs e DOCX serão limpos e divididos em trechos menores.

Esses trechos serão utilizados posteriormente para criar embeddings e realizar busca semântica.


In [18]:
import re


def clean_text(text: str) -> str:
    """
    Limpa espaços e quebras de linha desnecessárias.
    """

    if not isinstance(text, str):
        raise TypeError("O conteúdo precisa ser uma string.")

    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


In [19]:
dirty_text = """
    Python     e análise de dados.


    Experiência com modelos de linguagem.
"""

cleaned_text = clean_text(dirty_text)

print(cleaned_text)


Python e análise de dados.

 Experiência com modelos de linguagem.


In [20]:
assert "     " not in cleaned_text
assert cleaned_text.strip() == cleaned_text

print("Teste de limpeza aprovado.")


Teste de limpeza aprovado.


In [21]:
cleaned_sources = []

for source in all_sources:
    cleaned_source = source.copy()
    cleaned_source["content"] = clean_text(source["content"])
    cleaned_sources.append(cleaned_source)

print(f"Fontes processadas: {len(cleaned_sources)}")


Fontes processadas: 5


In [22]:
for source in cleaned_sources:
    print(f"Fonte: {source['source_name']}")
    print(f"Tipo: {source['source_type']}")
    print(f"Caracteres: {len(source['content'])}")
    print(f"Prévia: {source['content'][:200]}")
    print("=" * 80)


Fonte: Experiência acadêmica
Tipo: manual_text
Caracteres: 225
Prévia: Participei de uma iniciação científica envolvendo a comparação de modelos
 de inteligência artificial aplicados à correção de redações. Trabalhei com
 uma base de dados extensa, análise de resultados 
Fonte: 03_descricao_projeto_ficticio.docx
Tipo: docx
Caracteres: 807
Prévia: Descrição de Projeto — Documento de Teste
Projeto: Painel de acompanhamento de estudos
Contexto
Aplicação web fictícia criada para organizar metas de estudo, registrar sessões e sugerir revisões com b
Fonte: 02_relatorio_projeto_ficticio.pdf
Tipo: pdf
Caracteres: 870
Prévia: Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técnicas de processamento de linguagem natural para or
Fonte: 01_curriculo_candidato_ficticio.pdf
Tipo: pdf
Caracteres: 907
Prévia: Currículo de Demonstração
Marina Oliveira
Candidata fictícia para testes do Pathfolio
Objetivo
Bu

In [23]:
def split_text(
    text: str,
    chunk_size: int = 800,
    chunk_overlap: int = 120
) -> list[str]:
    """
    Divide um texto em trechos com sobreposição.

    A sobreposição ajuda a preservar o contexto entre dois trechos.
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size precisa ser maior que zero.")

    if chunk_overlap < 0:
        raise ValueError("chunk_overlap não pode ser negativo.")

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap precisa ser menor que chunk_size."
        )

    text = clean_text(text)

    if not text:
        return []

    chunks = []
    start = 0
    text_length = len(text)
    step = chunk_size - chunk_overlap

    while start < text_length:
        end = min(start + chunk_size, text_length)
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        if end >= text_length:
            break

        start += step

    return chunks


In [24]:
test_text = (
    "Python é uma linguagem utilizada em análise de dados. "
    "Machine Learning permite construir modelos preditivos. "
    "Modelos de linguagem podem processar textos e gerar respostas. "
    "A avaliação dos resultados é importante para entender limitações."
)

test_chunks = split_text(
    test_text,
    chunk_size=100,
    chunk_overlap=20
)

for index, chunk in enumerate(test_chunks):
    print(f"Trecho {index}:")
    print(chunk)
    print("-" * 50)


Trecho 0:
Python é uma linguagem utilizada em análise de dados. Machine Learning permite construir modelos pre
--------------------------------------------------
Trecho 1:
onstruir modelos preditivos. Modelos de linguagem podem processar textos e gerar respostas. A avalia
--------------------------------------------------
Trecho 2:
respostas. A avaliação dos resultados é importante para entender limitações.
--------------------------------------------------


In [25]:
assert len(test_chunks) > 1
assert all(len(chunk) > 0 for chunk in test_chunks)

print("Teste de divisão de texto aprovado.")


Teste de divisão de texto aprovado.


In [26]:
def create_chunks(
    sources: list[dict],
    chunk_size: int = 800,
    chunk_overlap: int = 120
) -> list[dict]:
    """
    Divide todas as fontes em trechos e preserva seus metadados.
    """

    chunks = []
    chunk_counter = 0

    for source in sources:
        source_chunks = split_text(
            source["content"],
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        for chunk_index, content in enumerate(source_chunks):
            chunks.append({
                "chunk_id": f"chunk-{chunk_counter:04d}",
                "content": content,
                "source_name": source["source_name"],
                "source_type": source["source_type"],
                "category": source["category"],
                "metadata": {
                    "page": source["metadata"].get("page"),
                    "file_name": source["metadata"].get("file_name"),
                    "chunk_index": chunk_index,
                    "source_created_at": source["metadata"].get(
                        "created_at"
                    )
                }
            })

            chunk_counter += 1

    return chunks


In [27]:
chunks = create_chunks(cleaned_sources)

print(f"Total de chunks criados: {len(chunks)}")


Total de chunks criados: 8


In [28]:
for chunk in chunks[:5]:
    print(f"ID: {chunk['chunk_id']}")
    print(f"Fonte: {chunk['source_name']}")
    print(f"Tipo: {chunk['source_type']}")
    print(f"Página: {chunk['metadata']['page']}")
    print(f"Índice: {chunk['metadata']['chunk_index']}")
    print(f"Conteúdo: {chunk['content'][:300]}")
    print("=" * 80)


ID: chunk-0000
Fonte: Experiência acadêmica
Tipo: manual_text
Página: None
Índice: 0
Conteúdo: Participei de uma iniciação científica envolvendo a comparação de modelos
 de inteligência artificial aplicados à correção de redações. Trabalhei com
 uma base de dados extensa, análise de resultados e comparação de métricas.
ID: chunk-0001
Fonte: 03_descricao_projeto_ficticio.docx
Tipo: docx
Página: None
Índice: 0
Conteúdo: Descrição de Projeto — Documento de Teste
Projeto: Painel de acompanhamento de estudos
Contexto
Aplicação web fictícia criada para organizar metas de estudo, registrar sessões e sugerir revisões com base no histórico do usuário.
Contribuições realizadas
Levantamento dos requisitos principais da apli
ID: chunk-0002
Fonte: 03_descricao_projeto_ficticio.docx
Tipo: docx
Página: None
Índice: 1
Conteúdo: ia do usuário.
Aprendizados
O projeto ajudou a transformar uma necessidade de usuário em funcionalidades menores e organizadas.
ID: chunk-0003
Fonte: 02_relatorio_projeto_ficti

In [29]:
assert len(chunks) > 0

for chunk in chunks:
    assert chunk["chunk_id"]
    assert chunk["content"]
    assert chunk["source_name"]
    assert chunk["source_type"] in {"manual_text", "pdf", "docx"}
    assert "metadata" in chunk
    assert "page" in chunk["metadata"]

print("Testes de chunks e metadados aprovados.")


Testes de chunks e metadados aprovados.


In [30]:
from collections import Counter

chunks_per_source = Counter(
    chunk["source_name"]
    for chunk in chunks
)

for source_name, quantity in chunks_per_source.items():
    print(f"{source_name}: {quantity} chunks")


Experiência acadêmica: 1 chunks
03_descricao_projeto_ficticio.docx: 2 chunks
02_relatorio_projeto_ficticio.pdf: 2 chunks
01_curriculo_candidato_ficticio.pdf: 3 chunks


In [31]:
def keyword_search(
    chunks: list[dict],
    query: str
) -> list[dict]:
    """
    Retorna chunks que contêm todas as palavras relevantes da busca.
    """

    query_terms = [
        term.lower()
        for term in re.findall(r"\w+", query)
        if len(term) > 2
    ]

    results = []

    for chunk in chunks:
        content = chunk["content"].lower()

        if all(term in content for term in query_terms):
            results.append(chunk)

    return results


In [32]:
results = keyword_search(
    chunks,
    "Python análise de dados"
)

print(f"Resultados encontrados: {len(results)}")

for result in results:
    print(result["source_name"])
    print(result["content"])
    print("-" * 80)


Resultados encontrados: 2
02_relatorio_projeto_ficticio.pdf
Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técnicas de processamento de linguagem natural para organizar feedbacks
acadêmicos por tema e prioridade, facilitando a análise por professores.
Atividades realizadas

Coleta e limpeza de exemplos textuais fictícios.

Definição de categorias para classificação dos feedbacks.

Criação de uma rotina de pré-processamento em Python.

Avaliação qualitativa dos resultados e análise de erros.

Registro das decisões e limitações do experimento.
Tecnologias e conceitos
Python, Pandas, classificação de texto, análise exploratória de dados, métricas de avaliação e Git.
Resultado
O experimento identificou situações em que textos ambíguos exigiam revisão
--------------------------------------------------------------------------------
01_curriculo_candidato_ficticio.pdf
Currículo de Demonstra

In [33]:
import json

chunks_output_path = "pathfolio_chunks_debug.json"

with open(chunks_output_path, "w", encoding="utf-8") as file:
    json.dump(chunks, file, ensure_ascii=False, indent=2)

print(f"Arquivo salvo: {chunks_output_path}")


Arquivo salvo: pathfolio_chunks_debug.json


## 4. Embeddings e busca semântica

Nesta etapa, cada chunk será convertido em um vetor numérico chamado embedding.

Chunks semanticamente parecidos ficarão próximos no espaço vetorial. Assim,
o Pathfolio poderá encontrar evidências relacionadas a uma vaga mesmo quando
as palavras utilizadas forem diferentes.


In [34]:
!pip -q install sentence-transformers scikit-learn


In [35]:
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


In [36]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(embedding_model_name)

print(f"Modelo carregado: {embedding_model_name}")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo carregado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [37]:
chunk_texts = [
    chunk["content"]
    for chunk in chunks
]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(f"Quantidade de embeddings: {len(chunk_embeddings)}")
print(f"Dimensão de cada embedding: {chunk_embeddings.shape[1]}")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Quantidade de embeddings: 8
Dimensão de cada embedding: 384


In [38]:
assert len(chunk_embeddings) == len(chunks)
assert chunk_embeddings.ndim == 2
assert chunk_embeddings.shape[1] > 0
assert np.isfinite(chunk_embeddings).all()

print("Teste de embeddings aprovado.")


Teste de embeddings aprovado.


In [39]:
embedding_norms = np.linalg.norm(chunk_embeddings, axis=1)

print(embedding_norms[:5])


[1.         0.99999994 1.         0.99999994 1.        ]


In [40]:
def semantic_search(
    query: str,
    chunks: list[dict],
    chunk_embeddings: np.ndarray,
    model: SentenceTransformer,
    top_k: int = 5
) -> list[dict]:
    """
    Retorna os chunks semanticamente mais próximos da consulta.
    """

    if not query or not query.strip():
        raise ValueError("A consulta não pode estar vazia.")

    if len(chunks) != len(chunk_embeddings):
        raise ValueError(
            "A quantidade de chunks precisa ser igual à quantidade de embeddings."
        )

    if top_k <= 0:
        raise ValueError("top_k precisa ser maior que zero.")

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    similarities = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    ranked_indices = np.argsort(similarities)[::-1]
    ranked_indices = ranked_indices[:min(top_k, len(chunks))]

    results = []

    for rank, index in enumerate(ranked_indices, start=1):
        result = chunks[index].copy()
        result["similarity_score"] = float(similarities[index])
        result["rank"] = rank
        results.append(result)

    return results


In [41]:
query = """
Experiência com Python, análise de dados e avaliação de modelos de inteligência artificial.
"""

semantic_results = semantic_search(
    query=query,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=5
)


In [42]:
for result in semantic_results:
    print(f"Rank: {result['rank']}")
    print(f"Similaridade: {result['similarity_score']:.4f}")
    print(f"Fonte: {result['source_name']}")
    print(f"Tipo: {result['source_type']}")
    print(f"Página: {result['metadata']['page']}")
    print(f"Conteúdo: {result['content']}")
    print("=" * 100)


Rank: 1
Similaridade: 0.6570
Fonte: 01_curriculo_candidato_ficticio.pdf
Tipo: pdf
Página: 2
Conteúdo: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre.

Python para análise de dados — curso livre.

Participação em grupo de estudos sobre Inteligência Artificial Generativa.
Informações adicionais
Interesse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e
desenvolvimento de produtos digitais.
Rank: 2
Similaridade: 0.6483
Fonte: 01_curriculo_candidato_ficticio.pdf
Tipo: pdf
Página: 1
Conteúdo: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
Rank: 3
Similaridade: 0.6235
Fonte: Experiênc

In [43]:
query_semantic = """
Vivência em experimentos com modelos computacionais capazes de trabalhar com linguagem humana.
"""

semantic_results_2 = semantic_search(
    query=query_semantic,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=5
)

for result in semantic_results_2:
    print(
        f"Rank {result['rank']} | "
        f"Score {result['similarity_score']:.4f} | "
        f"{result['source_name']}"
    )
    print(result["content"][:500])
    print("-" * 80)


Rank 1 | Score 0.6798 | 01_curriculo_candidato_ficticio.pdf
s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
--------------------------------------------------------------------------------
Rank 2 | Score 0.6141 | 01_curriculo_candidato_ficticio.pdf
Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre.

Python para análise de dados — curso livre.

Participação em grupo de estudos sobre Inteligência Artificial Generativa.
Informações adicionais
Interesse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e
desenvolvimento
----------------------------------------------------------------------

In [44]:
job_description = """
Estágio em Inteligência Artificial.

A pessoa candidata irá apoiar projetos de análise de dados e desenvolvimento
de soluções baseadas em modelos de linguagem. É desejável conhecimento em
Python, machine learning, processamento de linguagem natural, avaliação de
modelos e documentação de experimentos.
"""


In [45]:
job_results = semantic_search(
    query=job_description,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=5
)

for result in job_results:
    print(f"Rank: {result['rank']}")
    print(f"Score: {result['similarity_score']:.4f}")
    print(f"Fonte: {result['source_name']}")
    print(f"Página: {result['metadata']['page']}")
    print(f"Trecho: {result['content']}")
    print("=" * 100)


Rank: 1
Score: 0.7510
Fonte: 01_curriculo_candidato_ficticio.pdf
Página: 2
Trecho: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre.

Python para análise de dados — curso livre.

Participação em grupo de estudos sobre Inteligência Artificial Generativa.
Informações adicionais
Interesse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e
desenvolvimento de produtos digitais.
Rank: 2
Score: 0.7422
Fonte: 01_curriculo_candidato_ficticio.pdf
Página: 1
Trecho: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
Rank: 3
Score: 0.7071
Fonte: 01_curriculo_candidato_ficticio.pdf
Página: 1
Trecho: 

In [46]:
def format_evidence(results: list[dict]) -> list[dict]:
    """
    Formata os resultados para serem apresentados como evidências.
    """

    evidences = []

    for result in results:
        evidences.append({
            "evidence_id": result["chunk_id"],
            "rank": result["rank"],
            "similarity_score": result["similarity_score"],
            "source_name": result["source_name"],
            "source_type": result["source_type"],
            "page": result["metadata"].get("page"),
            "content": result["content"]
        })

    return evidences


In [47]:
evidences = format_evidence(job_results)

for evidence in evidences:
    print(evidence)


{'evidence_id': 'chunk-0007', 'rank': 1, 'similarity_score': 0.7510434985160828, 'source_name': '01_curriculo_candidato_ficticio.pdf', 'source_type': 'pdf', 'page': 2, 'content': 'Currículo de Demonstração — continuação\nHabilidades\nPython · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de\nDados · Git · GitHub · SQL básico\nCursos e atividades\n\x7f\nIntrodução a Machine Learning — curso livre.\n\x7f\nPython para análise de dados — curso livre.\n\x7f\nParticipação em grupo de estudos sobre Inteligência Artificial Generativa.\nInformações adicionais\nInteresse em aplicações de IA, avaliação de modelos, sistemas baseados em documentos e\ndesenvolvimento de produtos digitais.'}
{'evidence_id': 'chunk-0006', 'rank': 2, 'similarity_score': 0.7422292828559875, 'source_name': '01_curriculo_candidato_ficticio.pdf', 'source_type': 'pdf', 'page': 1, 'content': 's\nAssistente de estudos com processamento de linguagem natural\nProjeto experimental para ge

In [48]:
def filter_by_similarity(
    results: list[dict],
    minimum_score: float = 0.35
) -> list[dict]:
    """
    Mantém somente resultados acima da similaridade mínima.
    """

    return [
        result
        for result in results
        if result["similarity_score"] >= minimum_score
    ]


In [49]:
filtered_results = filter_by_similarity(
    job_results,
    minimum_score=0.35
)

print(f"Resultados antes do filtro: {len(job_results)}")
print(f"Resultados depois do filtro: {len(filtered_results)}")


Resultados antes do filtro: 5
Resultados depois do filtro: 5


In [50]:
unrelated_results = semantic_search(
    query="Experiência profissional em engenharia mecânica e manutenção industrial.",
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=5
)

for result in unrelated_results:
    print(
        f"{result['similarity_score']:.4f} | "
        f"{result['source_name']} | "
        f"{result['content'][:150]}"
    )


0.2714 | 03_descricao_projeto_ficticio.docx | Descrição de Projeto — Documento de Teste
Projeto: Painel de acompanhamento de estudos
Contexto
Aplicação web fictícia criada para organizar metas de 
0.2398 | 01_curriculo_candidato_ficticio.pdf | Currículo de Demonstração
Marina Oliveira
Candidata fictícia para testes do Pathfolio
Objetivo
Buscar oportunidade de estágio em Inteligência Artifici
0.2091 | 02_relatorio_projeto_ficticio.pdf | Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técn
0.2079 | Experiência acadêmica | Participei de uma iniciação científica envolvendo a comparação de modelos
 de inteligência artificial aplicados à correção de redações. Trabalhei com

0.2036 | 01_curriculo_candidato_ficticio.pdf | s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de


In [51]:
import json

semantic_results_path = "pathfolio_semantic_search_debug.json"

with open(semantic_results_path, "w", encoding="utf-8") as file:
    json.dump(
        job_results,
        file,
        ensure_ascii=False,
        indent=2
    )

print(f"Arquivo salvo: {semantic_results_path}")


Arquivo salvo: pathfolio_semantic_search_debug.json


In [52]:
assert len(job_results) > 0
assert len(job_results) <= 5

for index, result in enumerate(job_results, start=1):
    assert result["rank"] == index
    assert 0 <= result["similarity_score"] <= 1
    assert result["content"]
    assert result["source_name"]

for first, second in zip(job_results, job_results[1:]):
    assert first["similarity_score"] >= second["similarity_score"]

print("Testes de busca semântica aprovados.")


Testes de busca semântica aprovados.


## 5. Análise de competências e lacunas

Nesta etapa, o Pathfolio analisará uma descrição de vaga e verificará
quais competências possuem evidências nas informações fornecidas pelo candidato.

A aplicação não deverá considerar uma competência como comprovada apenas
porque ela aparece na descrição da vaga. Ela precisará estar relacionada
a algum trecho recuperado das fontes do candidato.


In [53]:
job_description = """
Estágio em Inteligência Artificial

A pessoa candidata irá apoiar projetos de análise de dados e desenvolvimento
de soluções baseadas em modelos de linguagem.

Requisitos:
- Conhecimento em Python;
- Noções de machine learning;
- Interesse em modelos de linguagem;
- Capacidade de analisar dados;
- Boa comunicação para apresentar resultados.

Diferenciais:
- Experiência com processamento de linguagem natural;
- Experiência com avaliação de modelos;
- Participação em projetos de pesquisa.
"""


In [54]:
skill_catalog = [
    "Python",
    "Machine Learning",
    "Análise de dados",
    "Modelos de linguagem",
    "Processamento de Linguagem Natural",
    "Avaliação de modelos",
    "Pesquisa",
    "Git",
    "GitHub",
    "SQL",
    "React",
    "JavaScript",
    "Comunicação"
]


In [55]:
skill_aliases = {
    "Python": [
        "python"
    ],
    "Machine Learning": [
        "machine learning",
        "aprendizado de máquina",
        "ml"
    ],
    "Análise de dados": [
        "análise de dados",
        "analise de dados",
        "análise dos dados",
        "analise dos dados",
        "data analysis"
    ],
    "Modelos de linguagem": [
        "modelos de linguagem",
        "modelo de linguagem",
        "large language model",
        "large language models",
        "llm",
        "llms"
    ],
    "Processamento de Linguagem Natural": [
        "processamento de linguagem natural",
        "natural language processing",
        "nlp"
    ],
    "Avaliação de modelos": [
        "avaliação de modelos",
        "avaliacao de modelos",
        "avaliação de modelagem",
        "model evaluation"
    ],
    "Pesquisa": [
        "pesquisa",
        "iniciação científica",
        "iniciacao cientifica",
        "investigação",
        "investigacao"
    ],
    "Git": [
        "git"
    ],
    "GitHub": [
        "github"
    ],
    "SQL": [
        "sql"
    ],
    "React": [
        "react"
    ],
    "JavaScript": [
        "javascript",
        "java script"
    ],
    "Comunicação": [
        "comunicação",
        "comunicacao",
        "apresentação de resultados",
        "apresentacao de resultados"
    ]
}


In [56]:
def find_skills_in_text(
    text: str,
    skill_aliases: dict[str, list[str]]
) -> list[str]:
    """
    Identifica competências com base em aliases conhecidos.
    """

    normalized_text = text.lower()
    found_skills = []

    for skill, aliases in skill_aliases.items():
        for alias in aliases:
            if alias.lower() in normalized_text:
                found_skills.append(skill)
                break

    return found_skills


In [57]:
job_skills = find_skills_in_text(
    job_description,
    skill_aliases
)

print("Competências identificadas na vaga:")
for skill in job_skills:
    print(f"- {skill}")


Competências identificadas na vaga:
- Python
- Machine Learning
- Análise de dados
- Modelos de linguagem
- Processamento de Linguagem Natural
- Avaliação de modelos
- Pesquisa
- Comunicação


In [58]:
def build_skill_evidence_index(
    chunks: list[dict],
    skill_aliases: dict[str, list[str]]
) -> dict[str, list[dict]]:
    """
    Cria um índice que relaciona competências aos chunks que as mencionam.
    """

    evidence_index = {
        skill: []
        for skill in skill_aliases
    }

    for chunk in chunks:
        found_skills = find_skills_in_text(
            chunk["content"],
            skill_aliases
        )

        for skill in found_skills:
            evidence_index[skill].append({
                "chunk_id": chunk["chunk_id"],
                "source_name": chunk["source_name"],
                "source_type": chunk["source_type"],
                "page": chunk["metadata"].get("page"),
                "content": chunk["content"]
            })

    return evidence_index


In [59]:
skill_evidence_index = build_skill_evidence_index(
    chunks=chunks,
    skill_aliases=skill_aliases
)


In [60]:
for skill, evidences in skill_evidence_index.items():
    if evidences:
        print(f"\n{skill}: {len(evidences)} evidência(s)")

        for evidence in evidences[:2]:
            print(f"- Fonte: {evidence['source_name']}")
            print(f"- Página: {evidence['page']}")
            print(f"- Trecho: {evidence['content'][:250]}")



Python: 3 evidência(s)
- Fonte: 02_relatorio_projeto_ficticio.pdf
- Página: 1
- Trecho: Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técnicas de processamento de linguagem natural para organizar feedbacks
acadêmicos por tema e prioridade
- Fonte: 01_curriculo_candidato_ficticio.pdf
- Página: 1
- Trecho: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.

Machine Learning: 1 evidência(s)
- Fonte: 01_curriculo_candidato_ficticio.pdf
- Página: 2
- Trecho: Currículo de Demonstração — continuação
Habilidades
Python · Machine Learning · Processamento de Linguagem Natural · Modelos de Linguagem · Análise de
Dados · Git · GitHub · SQL básico
Cursos e atividades

Introdução a Machine Learning — curso livre

Análise de d

In [61]:
def analyze_skills_and_gaps(
    job_description: str,
    chunks: list[dict],
    skill_aliases: dict[str, list[str]]
) -> dict:
    """
    Analisa competências da vaga e verifica evidências no perfil.
    """

    job_skills = find_skills_in_text(
        job_description,
        skill_aliases
    )

    evidence_index = build_skill_evidence_index(
        chunks,
        skill_aliases
    )

    found_skills = []
    missing_skills = []
    skill_analysis = []

    for skill in job_skills:
        evidences = evidence_index.get(skill, [])

        if evidences:
            status = "found"
            found_skills.append(skill)
        else:
            status = "not_found"
            missing_skills.append(skill)

        skill_analysis.append({
            "skill": skill,
            "status": status,
            "evidence_count": len(evidences),
            "evidence": evidences[:3]
        })

    return {
        "job_skills": job_skills,
        "found_skills": found_skills,
        "missing_skills": missing_skills,
        "skill_analysis": skill_analysis
    }


In [62]:
skills_analysis = analyze_skills_and_gaps(
    job_description=job_description,
    chunks=chunks,
    skill_aliases=skill_aliases
)


In [63]:
print("COMPETÊNCIAS ENCONTRADAS")
print("=" * 60)

for skill in skills_analysis["found_skills"]:
    print(f"- {skill}")

print("\nCOMPETÊNCIAS NÃO ENCONTRADAS")
print("=" * 60)

for skill in skills_analysis["missing_skills"]:
    print(f"- {skill}")



COMPETÊNCIAS ENCONTRADAS
- Python
- Machine Learning
- Análise de dados
- Modelos de linguagem
- Processamento de Linguagem Natural
- Avaliação de modelos
- Pesquisa

COMPETÊNCIAS NÃO ENCONTRADAS
- Comunicação


In [64]:
def calculate_skill_coverage(
    found_skills: list[str],
    job_skills: list[str]
) -> float:
    """
    Calcula a proporção de competências da vaga com evidências.
    """

    if not job_skills:
        return 0.0

    return len(found_skills) / len(job_skills)


In [65]:
coverage = calculate_skill_coverage(
    found_skills=skills_analysis["found_skills"],
    job_skills=skills_analysis["job_skills"]
)

print(f"Cobertura de competências: {coverage:.2%}")


Cobertura de competências: 87.50%


In [66]:
def retrieve_skill_evidence(
    skill: str,
    chunks: list[dict],
    chunk_embeddings: np.ndarray,
    model: SentenceTransformer,
    top_k: int = 3,
    minimum_score: float = 0.35
) -> list[dict]:
    """
    Recupera evidências semanticamente relacionadas a uma competência.
    """

    results = semantic_search(
        query=skill,
        chunks=chunks,
        chunk_embeddings=chunk_embeddings,
        model=model,
        top_k=top_k
    )

    return [
        result
        for result in results
        if result["similarity_score"] >= minimum_score
    ]


In [67]:
python_evidence = retrieve_skill_evidence(
    skill="Análise de dados",
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    top_k=3
)

for evidence in python_evidence:
    print(f"Score: {evidence['similarity_score']:.4f}")
    print(f"Fonte: {evidence['source_name']}")
    print(f"Conteúdo: {evidence['content'][:400]}")
    print("-" * 80)


Score: 0.5523
Fonte: Experiência acadêmica
Conteúdo: Participei de uma iniciação científica envolvendo a comparação de modelos
 de inteligência artificial aplicados à correção de redações. Trabalhei com
 uma base de dados extensa, análise de resultados e comparação de métricas.
--------------------------------------------------------------------------------
Score: 0.5411
Fonte: 01_curriculo_candidato_ficticio.pdf
Conteúdo: Currículo de Demonstração
Marina Oliveira
Candidata fictícia para testes do Pathfolio
Objetivo
Buscar oportunidade de estágio em Inteligência Artificial, Dados ou Desenvolvimento de Aplicações
Inteligentes.
Formação
Graduação em Sistemas de Informação
Universidade Exemplo — conclusão prevista: 2027
Experiência acadêmica
Projeto de iniciação científica — Avaliação de modelos de linguagem

Comparaç
--------------------------------------------------------------------------------
Score: 0.4900
Fonte: 02_relatorio_projeto_ficticio.pdf
Conteúdo: dados, métricas de avaliaç

In [68]:
def analyze_skill_hybrid(
    skill: str,
    chunks: list[dict],
    chunk_embeddings: np.ndarray,
    model: SentenceTransformer,
    skill_aliases: dict[str, list[str]],
    minimum_score: float = 0.35
) -> dict:
    """
    Analisa uma competência com evidência direta e semântica.

    Evidência direta confirma a competência.
    Evidência semântica isolada é marcada como candidata.
    """

    direct_evidence = []

    for chunk in chunks:
        found_skills = find_skills_in_text(
            chunk["content"],
            {skill: skill_aliases.get(skill, [])}
        )

        if skill in found_skills:
            direct_evidence.append({
                "chunk_id": chunk["chunk_id"],
                "source_name": chunk["source_name"],
                "source_type": chunk["source_type"],
                "page": chunk["metadata"].get("page"),
                "content": chunk["content"],
                "evidence_method": "keyword"
            })

    semantic_evidence = retrieve_skill_evidence(
        skill=skill,
        chunks=chunks,
        chunk_embeddings=chunk_embeddings,
        model=model,
        top_k=3,
        minimum_score=minimum_score
    )

    semantic_only_evidence = []
    direct_ids = {
        evidence["chunk_id"]
        for evidence in direct_evidence
    }

    for evidence in semantic_evidence:
        if evidence["chunk_id"] not in direct_ids:
            evidence_copy = evidence.copy()
            evidence_copy["evidence_method"] = "semantic_candidate"
            semantic_only_evidence.append(evidence_copy)

    if direct_evidence:
        status = "found"
        evidence_level = "direct"
    elif semantic_only_evidence:
        status = "candidate"
        evidence_level = "semantic_only"
    else:
        status = "not_found"
        evidence_level = "none"

    return {
        "skill": skill,
        "status": status,
        "evidence_level": evidence_level,
        "direct_evidence": direct_evidence[:5],
        "semantic_candidates": semantic_only_evidence[:5]
    }


In [69]:
def analyze_job_hybrid(
    job_description: str,
    chunks: list[dict],
    chunk_embeddings: np.ndarray,
    model: SentenceTransformer,
    skill_aliases: dict[str, list[str]],
    minimum_score: float = 0.35
) -> dict:
    """
    Analisa competências da vaga usando evidências diretas e semânticas.
    """

    job_skills = find_skills_in_text(
        job_description,
        skill_aliases
    )

    skill_results = []

    for skill in job_skills:
        result = analyze_skill_hybrid(
            skill=skill,
            chunks=chunks,
            chunk_embeddings=chunk_embeddings,
            model=model,
            skill_aliases=skill_aliases,
            minimum_score=minimum_score
        )

        skill_results.append(result)

    found_skills = [
        result["skill"]
        for result in skill_results
        if result["status"] == "found"
    ]

    candidate_skills = [
        result["skill"]
        for result in skill_results
        if result["status"] == "candidate"
    ]

    missing_skills = [
        result["skill"]
        for result in skill_results
        if result["status"] == "not_found"
    ]

    return {
        "job_skills": job_skills,
        "found_skills": found_skills,
        "candidate_skills": candidate_skills,
        "missing_skills": missing_skills,
        "skill_results": skill_results,
        "coverage": calculate_skill_coverage(
            found_skills,
            job_skills
        )
    }


In [71]:
hybrid_analysis = analyze_job_hybrid(
    job_description=job_description,
    chunks=chunks,
    chunk_embeddings=chunk_embeddings,
    model=embedding_model,
    skill_aliases=skill_aliases,
    minimum_score=0.35
)


In [72]:
print(hybrid_analysis.keys())


dict_keys(['job_skills', 'found_skills', 'candidate_skills', 'missing_skills', 'skill_results', 'coverage'])


In [73]:
print(f"Cobertura: {hybrid_analysis['coverage']:.2%}")

print("\nEncontradas:")
for skill in hybrid_analysis["found_skills"]:
    print(f"- {skill}")

print("\nCandidatas:")
for skill in hybrid_analysis["candidate_skills"]:
    print(f"- {skill}")

print("\nNão encontradas:")
for skill in hybrid_analysis["missing_skills"]:
    print(f"- {skill}")


Cobertura: 87.50%

Encontradas:
- Python
- Machine Learning
- Análise de dados
- Modelos de linguagem
- Processamento de Linguagem Natural
- Avaliação de modelos
- Pesquisa

Candidatas:
- Comunicação

Não encontradas:


In [74]:
for result in hybrid_analysis["skill_results"]:
    print(f"Competência: {result['skill']}")
    print(f"Status: {result['status']}")
    print(f"Nível de evidência: {result['evidence_level']}")

    print("\nEvidências diretas:")
    for evidence in result["direct_evidence"]:
        print(f"- Fonte: {evidence['source_name']}")
        print(f"- Página: {evidence['page']}")
        print(f"- Trecho: {evidence['content'][:500]}")

    print("\nCandidatas semânticas:")
    for evidence in result["semantic_candidates"]:
        print(f"- Score: {evidence['similarity_score']:.4f}")
        print(f"- Fonte: {evidence['source_name']}")
        print(f"- Trecho: {evidence['content'][:500]}")

    print("=" * 100)


Competência: Python
Status: found
Nível de evidência: direct

Evidências diretas:
- Fonte: 02_relatorio_projeto_ficticio.pdf
- Página: 1
- Trecho: Relatório de Projeto — Documento de Teste
Projeto: Classificação de feedbacks acadêmicos
Responsável: Marina Oliveira
Resumo
O projeto investigou técnicas de processamento de linguagem natural para organizar feedbacks
acadêmicos por tema e prioridade, facilitando a análise por professores.
Atividades realizadas

Coleta e limpeza de exemplos textuais fictícios.

Definição de categorias para classificação dos feedbacks.

Criação de uma rotina de pré-processamento em Python.

Avaliação qualitat
- Fonte: 01_curriculo_candidato_ficticio.pdf
- Página: 1
- Trecho: s
Assistente de estudos com processamento de linguagem natural
Projeto experimental para gerar perguntas de revisão e resumos a partir de materiais de estudo. Utilização
de Python, processamento de texto e modelos de linguagem.
- Fonte: 01_curriculo_candidato_ficticio.pdf
- Página: 2


In [75]:
for result in hybrid_analysis["skill_results"]:
    print(
        f"{result['skill']} | "
        f"status={result['status']} | "
        f"evidência={result['evidence_level']} | "
        f"diretas={len(result['direct_evidence'])} | "
        f"candidatas={len(result['semantic_candidates'])}"
    )


Python | status=found | evidência=direct | diretas=3 | candidatas=0
Machine Learning | status=found | evidência=direct | diretas=1 | candidatas=2
Análise de dados | status=found | evidência=direct | diretas=1 | candidatas=3
Modelos de linguagem | status=found | evidência=direct | diretas=3 | candidatas=1
Processamento de Linguagem Natural | status=found | evidência=direct | diretas=4 | candidatas=0
Avaliação de modelos | status=found | evidência=direct | diretas=2 | candidatas=2
Pesquisa | status=found | evidência=direct | diretas=2 | candidatas=2
Comunicação | status=candidate | evidência=semantic_only | diretas=0 | candidatas=1


In [76]:
def show_missing_skills(analysis: dict) -> None:
    missing_skills = analysis["missing_skills"]

    if not missing_skills:
        print("Nenhuma lacuna foi identificada.")
        return

    print("Competências sem evidência direta:")
    for skill in missing_skills:
        print(f"- {skill}")


In [77]:
show_missing_skills(hybrid_analysis)


Nenhuma lacuna foi identificada.


In [78]:
def build_skill_report(analysis: dict) -> dict:
    return {
        "coverage": analysis["coverage"],
        "job_skills": analysis["job_skills"],
        "found_skills": analysis["found_skills"],
        "candidate_skills": analysis["candidate_skills"],
        "missing_skills": analysis["missing_skills"],
        "summary": {
            "total_skills": len(analysis["job_skills"]),
            "found_count": len(analysis["found_skills"]),
            "candidate_count": len(analysis["candidate_skills"]),
            "missing_count": len(analysis["missing_skills"])
        }
    }


skill_report = build_skill_report(hybrid_analysis)
skill_report


{'coverage': 0.875,
 'job_skills': ['Python',
  'Machine Learning',
  'Análise de dados',
  'Modelos de linguagem',
  'Processamento de Linguagem Natural',
  'Avaliação de modelos',
  'Pesquisa',
  'Comunicação'],
 'found_skills': ['Python',
  'Machine Learning',
  'Análise de dados',
  'Modelos de linguagem',
  'Processamento de Linguagem Natural',
  'Avaliação de modelos',
  'Pesquisa'],
 'candidate_skills': ['Comunicação'],
 'missing_skills': [],
 'summary': {'total_skills': 8,
  'found_count': 7,
  'candidate_count': 1,
  'missing_count': 0}}

In [79]:
import json

with open("pathfolio_skill_report.json", "w", encoding="utf-8") as file:
    json.dump(
        skill_report,
        file,
        ensure_ascii=False,
        indent=2
    )

print("Relatório salvo.")


Relatório salvo.


In [80]:
all_analyzed_skills = (
    hybrid_analysis["found_skills"]
    + hybrid_analysis["candidate_skills"]
    + hybrid_analysis["missing_skills"]
)

assert set(all_analyzed_skills) == set(
    hybrid_analysis["job_skills"]
)

assert not (
    set(hybrid_analysis["found_skills"])
    & set(hybrid_analysis["missing_skills"])
)

assert not (
    set(hybrid_analysis["candidate_skills"])
    & set(hybrid_analysis["missing_skills"])
)

print("Teste de consistência da análise aprovado.")


Teste de consistência da análise aprovado.
